# Is the splitting-tree MZI mesh unitary?

**The structure (from the two mirror-image diagrams).** The chip realizes a target matrix by its SVD
$$A = U\,\Sigma\,V,$$
where $U$ and $V$ are two **mirror-symmetric** $6\times6$ MZI meshes (the "beginning" and "end") and
$\Sigma$ is the **middle attenuator stage** (the diagonal singular values). Inputs enter through a
splitting tree; outputs are read at photodiodes.

**The question.** Some outputs are *attenuated* — the $\Sigma$ stage dumps part of the light to unused
ports. That is effectively a **compression / partial trace** of a larger lossless network, so the
input→read map may *not* be unitary, even though $U$ and $V$ are mirror-symmetric. We check this
symbolically.

**The MZI equation** (two arm phase shifters $\theta_1,\theta_2$ between two 50:50 couplers
$C=\tfrac1{\sqrt2}\begin{psmallmatrix}1&i\\ i&1\end{psmallmatrix}$, from `src/mzi.py`):
$$M = C\,\mathrm{diag}(e^{i\theta_1},e^{i\theta_2})\,C
   = i\,e^{i(\theta_1+\theta_2)/2}\begin{pmatrix}\sin\Delta&\cos\Delta\\\cos\Delta&-\sin\Delta\end{pmatrix},
   \quad \Delta=\tfrac{\theta_1-\theta_2}{2}.$$
The 1-in/1-out **attenuator** (other output dumped) has transmission $t=M_{00}$, amplitude $|\sin\Delta|$.

In [1]:
import sympy as sp
sp.init_printing()
I = sp.I
th1, th2 = sp.symbols('theta1 theta2', real=True)
C = sp.Matrix([[1, I], [I, 1]]) / sp.sqrt(2)            # 50:50 coupler
P = sp.diag(sp.exp(I*th1), sp.exp(I*th2))               # two arm phase shifters
M = sp.simplify(C * P * C)                              # the MZI transfer matrix
M

⎡   ⅈ⋅θ₁    ⅈ⋅θ₂      ⎛ ⅈ⋅θ₁    ⅈ⋅θ₂⎞⎤
⎢  ℯ       ℯ        ⅈ⋅⎝ℯ     + ℯ    ⎠⎥
⎢  ───── - ─────    ─────────────────⎥
⎢    2       2              2        ⎥
⎢                                    ⎥
⎢  ⎛ ⅈ⋅θ₁    ⅈ⋅θ₂⎞      ⅈ⋅θ₁    ⅈ⋅θ₂ ⎥
⎢ⅈ⋅⎝ℯ     + ℯ    ⎠     ℯ       ℯ     ⎥
⎢─────────────────   - ───── + ───── ⎥
⎣        2               2       2   ⎦

## 1. A single MZI (2-in / 2-out) is *exactly* unitary

It is a product of unitaries (coupler · phase · coupler), so $M^\dagger M = \mathbb{1}$ identically.

In [2]:
MhM = sp.simplify(M.H * M)
print("M^H M - I =")
sp.pprint(sp.simplify(MhM - sp.eye(2)))
assert sp.simplify(MhM - sp.eye(2)) == sp.zeros(2)
print("\n=> the 2x2 MZI is unitary for all theta1, theta2.")

M^H M - I =
⎡0  0⎤
⎢    ⎥
⎣0  0⎦

=> the 2x2 MZI is unitary for all theta1, theta2.


## 2. Used as an *attenuator* it is **not** unitary — it leaks power

Dump output port 1; the kept transmission is $t=M_{00}$ with $|t|^2=\sin^2\Delta$. Stacking such
attenuators on the diagonal gives the $\Sigma$ stage $\Sigma=\mathrm{diag}(\sin\Delta_1,\dots)$. The
deficiency $\Sigma^\dagger\Sigma-\mathbb1=-\mathrm{diag}(\cos^2\Delta_i)$ is exactly the **dumped power**.

In [3]:
t = M[0, 0]
mag2 = sp.simplify(sp.re(t)**2 + sp.im(t)**2)
print("|t|^2 =", mag2, "  ==  sin^2(Delta)?", sp.simplify(mag2 - sp.sin((th1-th2)/2)**2) == 0)

d1, d2 = sp.symbols('Delta1 Delta2', real=True)
Sigma = sp.diag(sp.sin(d1), sp.sin(d2))
print("\nSigma^H Sigma - I =")
sp.pprint(sp.simplify(Sigma.H * Sigma - sp.eye(2)))
print("\n=> nonzero (= -cos^2 Delta_i = leaked power), so Sigma is a CONTRACTION, not unitary.")

|t|^2 = 1/2 - cos(theta1 - theta2)/2   ==  sin^2(Delta)? True

Sigma^H Sigma - I =
⎡    2               ⎤
⎢-cos (Δ₁)      0    ⎥
⎢                    ⎥
⎢               2    ⎥
⎣    0      -cos (Δ₂)⎦

=> nonzero (= -cos^2 Delta_i = leaked power), so Sigma is a CONTRACTION, not unitary.


## 3. The two halves $U,V$ (couplers + phase shifters only) *are* unitary

Embedding a $2\times2$ unitary into the $N$-mode identity is unitary, and a product of unitaries is
unitary — so any lossless MZI mesh is unitary. Confirmed numerically on the real 6-mode Clements mesh
from `src/mzi.py`.

In [4]:
import sys, numpy as np
sys.path.insert(0, '..'); sys.path.insert(0, '.')
from src import mzi
N = 6
layers = mzi.clements_layers(N)
rng = np.random.default_rng(0)
params = {(li, m): (rng.uniform(0, 2*np.pi), rng.uniform(0, 2*np.pi))
          for li, L in enumerate(layers) for m in L}
U = mzi.mesh_unitary(N, params, layers)
print("6-mode mesh U^H U == I ?", np.allclose(U.conj().T @ U, np.eye(N)))
print("max|U^H U - I| =", np.abs(U.conj().T @ U - np.eye(N)).max())

6-mode mesh U^H U == I ? True
max|U^H U - I| = 2.220446049250313e-15


## 4. The composite $A=U\,\Sigma\,V$: **symmetric, but not unitary**

With the mirror symmetry the diagram shows ($V=U^{\mathsf T}$), $A=U\,\Sigma\,U^{\mathsf T}$ is
**symmetric** — which is the user's observation. But symmetry $\ne$ unitarity: the **singular values of
$A$ are exactly the attenuations $|\sin\Delta_i|\le 1$**, so $A^\dagger A=\mathbb1$ holds *iff* every
$\sin\Delta_i=\pm1$, i.e. $\Sigma=\mathbb1$ (no attenuation).

In [5]:
a = sp.symbols('alpha', real=True)
Uu = sp.Matrix([[sp.cos(a), -sp.sin(a)], [sp.sin(a), sp.cos(a)]])   # a real 2-mode unitary
A = Uu * Sigma * Uu.T
print("A symmetric (A - A^T)?"); sp.pprint(sp.simplify(A - A.T))
print("\nsingular values of A:", [sp.simplify(s) for s in A.singular_values()])
print("\nA^H A == I in general?", sp.simplify(A.H*A - sp.eye(2)) == sp.zeros(2))
print("A^H A == I when sin(Delta)=1 (Sigma=I)?",
      sp.simplify((A.H*A).subs({d1: sp.pi/2, d2: sp.pi/2}) - sp.eye(2)) == sp.zeros(2))

A symmetric (A - A^T)?
⎡0  0⎤
⎢    ⎥
⎣0  0⎦



singular values of A: [Abs(sin(Delta1)), Abs(sin(Delta2))]



A^H A == I in general? False
A^H A == I when sin(Delta)=1 (Sigma=I)? True


## 5. Why, formally: $A$ is a *compression of a unitary*

The full physical device (system ports **plus** the dumped ports and monitor photodiodes) is lossless
passive linear optics, so its scattering matrix $S$ is unitary on **all** ports. The "transfer matrix"
$A$ we use is the block of $S$ from the used inputs to the read outputs — a **submatrix of a unitary**.
Such a compression is always a contraction: $A^\dagger A\preceq\mathbb1$, all singular values $\le1$,
with equality only if no light reaches the dropped ports. Dropping ports (the $\Sigma$ dump) is exactly
the partial-trace-like restriction. Demonstration: take a random unitary, keep a sub-block.

In [6]:
Qr = np.linalg.qr(rng.normal(size=(5,5)) + 1j*rng.normal(size=(5,5)))[0]
print("5x5 Q unitary?", np.allclose(Qr.conj().T @ Qr, np.eye(5)))
block = Qr[:3, :3]                     # keep 3 inputs, 3 outputs; drop 2 ports
s = np.linalg.svd(block, compute_uv=False)
print("singular values of the 3x3 sub-block:", np.round(s, 3))
print("all <= 1 (contraction):", bool(s.max() <= 1 + 1e-12), " strictly < 1 somewhere:", bool(s.min() < 1))

5x5 Q unitary? True
singular values of the 3x3 sub-block: [1.    0.759 0.616]
all <= 1 (contraction): True  strictly < 1 somewhere: True


## Conclusion

| object | unitary? | why |
|---|---|---|
| single MZI (2-in/2-out) | **yes** (exact) | product of couplers + phases; $M^\dagger M=\mathbb1$ |
| the $U,V$ halves (lossless mesh) | **yes** | product/embedding of unitaries |
| attenuator / $\Sigma$ stage | **no** | dumps power; $\Sigma^\dagger\Sigma-\mathbb1=-\mathrm{diag}(\cos^2\Delta_i)$ |
| composite $A=U\Sigma V$ | **no** (in general) | singular values $=|\sin\Delta_i|\le1$; unitary iff $\Sigma=\mathbb1$ |
| $A=U\Sigma U^{\mathsf T}$ (mirror-symmetric) | **no**, but **symmetric** | symmetry $\ne$ unitarity |

**Your instinct was right.** Attenuated outputs = a compression (partial trace) of a larger unitary,
so the input→read map is **sub-unitary**, not unitary. The mirror symmetry makes $A$ **symmetric**
($A=A^{\mathsf T}$) — it does **not** make it unitary.

And this is **by design**: a purely unitary device could only ever compute unitary maps; the $\Sigma$
attenuators are what let it realize a *general* matrix $A$ (any singular values $\le1$) via SVD. Two
practical consequences: (1) **power is not conserved** at the read ports — you must account for the
dumped/monitor ports (homodyne uses exactly those taps); (2) **inversion needs the pseudo-inverse**,
since singular values can be $0$ (a fully-closed $\Sigma$ element kills that mode).